# Clustering des séries temporelles de consommation énergétique des bâtiments individuels

## Objectif du notebook

L'objectif de ce notebook est d'analyser les séries temporelles de consommation énergétique des bâtiments individuels afin d'identifier des **jours types de consommation**.

L'idée est de regrouper les journées présentant des comportements similaires en appliquant une méthode de **clustering non supervisé**. Ces profils journaliers caractéristiques permettront ensuite de mieux comprendre les habitudes de consommation des bâtiments et d'identifier différents comportements énergétiques.

---

## Approche suivie

La méthodologie est organisée en plusieurs étapes :

1. **Chargement des séries temporelles**
   - Lecture des fichiers de consommation énergétique des bâtiments individuels.
   - Sélection de la variable de consommation électrique étudiée.
   - Vérification de la fréquence temporelle des données.

2. **Transformation des séries temporelles en profils journaliers**
   - Les séries annuelles (pas de temps de 15 minutes) sont restructurées sous forme de matrices :
   Chaque ligne représente alors un profil de consommation sur une journée.

3. **Prétraitement des profils**
   - Normalisation des profils journaliers afin de comparer les formes de consommation indépendamment du niveau énergétique absolu.
   - Cette étape permet de détecter des comportements similaires même lorsque les bâtiments ont des consommations différentes.

4. **Détermination du nombre optimal de clusters**
   - Plusieurs valeurs du nombre de groupes sont testées.
   - Le score de silhouette est utilisé pour mesurer la qualité de séparation des clusters.

5. **Application du clustering**
   - Utilisation de l'algorithme **K-Means** pour regrouper les jours présentant des profils similaires.
   - Chaque cluster représente un **jour type de consommation**.

6. **Analyse et visualisation des résultats**
   - Visualisation des centroïdes des clusters correspondant aux journées représentatives.
   - Analyse de la fréquence d'apparition de chaque type de journée.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from kneed import KneeLocator
from sklearn.decomposition import PCA
import glob


ROOT = Path().resolve().parent.parent



DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES        = ROOT / 'reports' / 'figures'


df = pd.read_parquet(DATA_RAW / "347201-0.parquet")
df.head()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

dff = (
 	pd.read_parquet(DATA_RAW / "347201-0.parquet")
   	.assign(timestamp=lambda x: x["timestamp"] - pd.Timedelta("15m"))
   	.set_index("timestamp")
   	.loc[:, lambda x:
x.columns.str.match(r"out\.electricity\..*\.energy_consumption\.\.kwh")]
   	.loc[:, lambda x: x.ne(0).any(axis=0)]
   	.rename(columns=lambda x: x[16:-24])
   	.drop(columns="net")
)

dfh = dff.resample("h").sum()
dfd = dfh.resample("D").sum()

fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
dfd.drop(columns="total").plot(ax=ax, kind="area", cmap="tab20")
plt.legend(loc="upper right")
plt.show()
plt.close(fig)

piv = dfh.pivot_table(index=dfh.index.normalize(),
columns=dfh.index.hour, values="total")
cmap = (piv.index.day_of_week // 5).map({0: "tab:blue", 1: "tab:red"})
fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
piv.T.plot(ax=ax, color=cmap, alpha=0.1, legend=False)
plt.show()


## Extraire les jours

In [ ]:
df.shape

In [ ]:
COL = "out.electricity.total.energy_consumption..kwh"

# Série de consommation
ts = df[COL].values

# Nombre de mesures par jour (15 min)
steps_per_day = 96

# Récupération des timestamps
if "timestamp" in df.columns:
    dates = pd.to_datetime(df["timestamp"])

elif isinstance(df.index, pd.DatetimeIndex):
    dates = pd.to_datetime(df.index)

else:
    dates = pd.date_range( start="2018-01-01",periods=len(df),freq="15min")

# Nombre de jours complets
n_days = len(ts) // steps_per_day

# Garder uniquement les jours complets
ts = ts[: n_days * steps_per_day]
dates = dates[: n_days * steps_per_day]

# Transformation en matrice :
# lignes = jours
# colonnes = pas de temps dans la journée
days = ts.reshape(n_days, steps_per_day)

# Une date par jour
day_dates = (
    pd.Series(dates)
    .groupby(pd.Series(np.arange(len(dates))) // steps_per_day)
    .first()
    .dt.normalize()
    .values
)

print("Nombre de jours :", n_days)
print("Shape matrice jours :", days.shape)
print("Première date :", day_dates[0])

In [ ]:
# Supprimer les jours avec NaN
mask_valid = ~np.isnan(days).any(axis=1)
print(f"Jours invalides supprimés : {(~mask_valid).sum()} / {len(days)}")

days_clean = days[mask_valid]
day_dates_clean = day_dates[mask_valid]

## Normalisation des profils journaliers

In [ ]:
# Option A : StandardScaler global (garde le niveau d'amplitude)
scaler_global = StandardScaler()
days_scaled_global = scaler_global.fit_transform(days_clean)

# Option B : Normalisation par jour (forme du profil, z-score intra-jour)
day_mean = days_clean.mean(axis=1, keepdims=True)
day_std  = days_clean.std(axis=1, keepdims=True)
day_std[day_std == 0] = 1e-8  # éviter division par 0 (jours plats)

days_scaled_shape = (days_clean - day_mean) / day_std

# --> on choisit l'option B pour la suite (jours types = formes de profils)
X = days_scaled_shape
Y = days_scaled_global 


**Option A — `StandardScaler` global (variable `Y`, "amplitude")**
Centre-réduit chaque pas horaire (chaque colonne de la matrice jours × heures) à travers TOUS
les jours : pour une heure donnée, on compare la consommation de ce jour à la moyenne/écart-type
de cette même heure sur l'ensemble des jours. Les écarts de niveau global entre jours (un jour
très consommateur vs un jour creux) sont **préservés** → cette normalisation capte surtout le
niveau/l'amplitude de consommation, pas la forme de la courbe.

**Option B — z-score intra-jour (variable `X`, "forme")**
Centre-réduit **chaque jour individuellement** (ligne), en soustrayant sa propre moyenne et en
divisant par son propre écart-type. Le niveau absolu du jour disparaît complètement : seule reste
la **forme relative** du profil (où se situent les pics et les creux dans la journée). Deux jours
de niveaux très différents mais de même allure (ex. un jour froid et un jour chaud avec le même
pic le soir) donnent des vecteurs quasi identiques une fois normalisés.

→ On choisit **B (`X`)** pour regrouper les journées par forme de profil ("jours types"), et on
garde **A (`Y`)** en parallèle pour un clustering par niveau de consommation. Les deux sont
comparés côte à côte dans la suite du notebook (coude/silhouette, PCA, visualisations).

## Choix du nombre de clusters (méthode du coude + silhouette)

In [ ]:
def plot_cluster_profiles(X, title1,title2):

    k_range = range(2, 15)
    inertias = []
    silhouettes = []

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X)
        inertias.append(km.inertia_)
        sil = silhouette_score(X, labels, sample_size=5000, random_state=42)  # sample_size pour accélérer si beaucoup de jours
        silhouettes.append(sil)
        print(f"k={k:2d} | inertia={km.inertia_:10.1f} | silhouette={sil:.4f}")
    k_opt = KneeLocator(
    list(k_range),
    inertias,
    curve="convex",
    direction="decreasing"
    ).elbow

    print(f"K optimal selon la méthode du coude : {k_opt}")    

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(list(k_range), inertias, marker='o')
    axes[0].set_xlabel("k")
    axes[0].set_ylabel("Inertie")
    axes[0].set_title(title1)

    axes[1].plot(list(k_range), silhouettes, marker='o', color='orange')
    axes[1].set_xlabel("k")
    axes[1].set_ylabel("Silhouette score")
    axes[1].set_title(title2)

    plt.tight_layout()
    plt.show()
    return k_opt

k_forme=plot_cluster_profiles(X, "Méthode du coude (inertie) - Clustering des jours types", "Silhouette score - Clustering des jours types")
k_amplet=plot_cluster_profiles(Y, "Méthode du coude (inertie) - Clustering des jours types (niveau d'amplitude)", "Silhouette score - Clustering des jours types (niveau d'amplitude)")


## Clustering final avec le k choisi

In [ ]:
def cluster_days(k, X_input=X, dates_input=day_dates_clean):

    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    cluster_labels = kmeans.fit_predict(X_input)

    results = pd.DataFrame({
        "date": dates_input,
        "cluster": cluster_labels
    })

    results["weekday"] = results["date"].dt.day_name()
    results["is_weekend"] = results["date"].dt.dayofweek >= 5
    results["month"] = results["date"].dt.month

    print(results["cluster"].value_counts().sort_index())

    return kmeans, cluster_labels, results
kmeans_forme, cluster_labels_forme, results_forme =cluster_days(k_forme)
kmeans_amplet, cluster_labels_amplet, results_amplet=cluster_days(k_amplet)
print(results_forme)
print(results_amplet)

## Visualisation : profil moyen par cluster (courbes)

In [ ]:

def plot_interactive_cluster_profiles(cluster_labels, k_final, titre, days_input=days_clean):

    fig = go.Figure()

    hours = np.linspace(0, 24, steps_per_day)

    # Traces
    for c in range(k_final):
        mask = cluster_labels == c

        cluster_profiles = days_input[mask]
        mean_profile = cluster_profiles.mean(axis=0)
        std_profile = cluster_profiles.std(axis=0)

        # Courbe moyenne
        fig.add_trace(
            go.Scatter(
                x=hours,
                y=mean_profile,
                mode="lines",
                name=f"Cluster {c} (n={mask.sum()})",
                visible=(c == 0),
                line=dict(width=3)
            )
        )

        # Zone ±1σ
        fig.add_trace(
            go.Scatter(
                x=np.concatenate([hours, hours[::-1]]),
                y=np.concatenate([
                    mean_profile + std_profile,
                    (mean_profile - std_profile)[::-1]
                ]),
                fill="toself",
                mode="lines",
                opacity=0.15,
                name=f"Std Cluster {c}",
                visible=(c == 0),
                showlegend=False,
                hoverinfo="skip"
            )
        )

    # Boutons
    buttons = []

    for c in range(k_final):
        visibility = []

        for i in range(k_final):
            visibility += [i == c, i == c]

        buttons.append(
            dict(
                label=f"Cluster {c}",
                method="update",
                args=[
                    {"visible": visibility},
                    {"title": f"{titre} - Cluster {c}"}
                ]
            )
        )

    # Afficher tous les clusters
    buttons.append(
        dict(
            label="Tous les clusters",
            method="update",
            args=[
                {"visible": [True] * (2 * k_final)},
                {"title": f"{titre} (k={k_final})"}
            ]
        )
    )

    fig.update_layout(
        title=titre,
        xaxis_title="Heure de la journée",
        yaxis_title="Consommation électrique (kWh)",
        template="plotly_white",
        width=1000,
        height=600,
        updatemenus=[
            dict(
                buttons=buttons,
                direction="down",
                x=1.05,
                y=1,
                showactive=True
            )
        ]
    )

    fig.show()

    return 0

plot_interactive_cluster_profiles(
    cluster_labels=cluster_labels_amplet,
    k_final=k_amplet,
    titre="Profil journalier par rapport à la quantite d'energie consommée"
)
plot_interactive_cluster_profiles(
    cluster_labels=cluster_labels_forme,
    k_final=k_forme,
    titre="Profil journalier par rapport à la forme d'energie consommée"
)

## Décomposition par poste de consommation (au lieu de la courbe totale)

Les visualisations précédentes montrent la courbe **totale** moyenne par cluster. Pour comprendre
*ce qui* compose cette courbe, on décompose ici chaque cluster par poste de consommation électrique
(chauffage/HVAC, eau chaude, éclairage, électroménager, etc. — colonnes
`out.electricity.<poste>.energy_consumption..kwh` du fichier brut), au lieu d'agréger en une seule
courbe totale.

In [ ]:
ENDUSE_PATTERN = r"out\.electricity\..*\.energy_consumption\.\.kwh"

def extract_enduse_days(df_raw, mask_valid, n_days, steps_per_day=steps_per_day):
    """Reshape chaque poste de consommation électrique (hors 'total'/'net') en matrice
    (jours, pas de temps), avec le même découpage/masquage que la courbe totale (days_clean),
    pour rester aligné sur les labels de cluster calculés sur cette dernière."""
    enduse = (
        df_raw
        .loc[:, lambda x: x.columns.str.match(ENDUSE_PATTERN)]
        .loc[:, lambda x: x.ne(0).any(axis=0)]
        .rename(columns=lambda x: x[16:-24])
        .drop(columns=["net", "total"], errors="ignore")
    )

    enduse_days = {}
    for col in enduse.columns:
        vals = enduse[col].values[: n_days * steps_per_day]
        enduse_days[col] = vals.reshape(n_days, steps_per_day)[mask_valid]

    return enduse_days


def plot_cluster_profiles_by_enduse(enduse_days, cluster_labels, k_final, titre, steps_per_day=steps_per_day):
    """Pour chaque cluster, courbe moyenne empilée PAR POSTE de consommation
    (chauffage, eau chaude, éclairage, électroménager...) au lieu de la seule courbe totale."""
    hours = np.linspace(0, 24, steps_per_day)
    postes = list(enduse_days.keys())

    n_cols = min(3, k_final)
    n_rows = int(np.ceil(k_final / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).flatten()

    for c in range(k_final):
        mask = cluster_labels == c
        mean_by_poste = {poste: enduse_days[poste][mask].mean(axis=0) for poste in postes}
        stacked = pd.DataFrame(mean_by_poste, index=hours)
        stacked.plot(kind="area", ax=axes[c], cmap="tab20", legend=(c == 0), linewidth=0)
        axes[c].set_title(f"Cluster {c} (n={mask.sum()})")
        axes[c].set_xlabel("Heure")
        axes[c].set_ylabel("kWh")

    for ax in axes[k_final:]:
        ax.set_visible(False)

    fig.suptitle(titre, fontsize=14)
    plt.tight_layout()
    plt.show()


enduse_days = extract_enduse_days(df, mask_valid, n_days)
print("Postes de consommation détectés (347201-0) :", list(enduse_days.keys()))

plot_cluster_profiles_by_enduse(
    enduse_days, cluster_labels_forme, k_forme,
    "347201-0 — Décomposition par poste — profils de forme"
)
plot_cluster_profiles_by_enduse(
    enduse_days, cluster_labels_amplet, k_amplet,
    "347201-0 — Décomposition par poste — profils d'amplitude"
)

In [ ]:
def plot_cluster_calendar_composition(results, titre):

    # Répartition semaine vs weekend
    comp_weekend = pd.crosstab(
        results["cluster"], 
        results["is_weekend"], 
        normalize="index"
    ) * 100

    comp_weekend.columns = ["Semaine (%)", "Weekend (%)"]

    print("Répartition semaine / weekend :")
    print(comp_weekend.round(1))
    print()

    # Répartition par mois
    comp_month = pd.crosstab(
        results["cluster"], 
        results["month"], 
        normalize="index"
    ) * 100

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Weekend
    comp_weekend.plot(
        kind="bar",
        stacked=True,
        ax=axes[0],
        colormap="Set2"
    )

    axes[0].set_title("Répartition semaine / weekend par cluster")
    axes[0].set_ylabel("%")
    axes[0].set_xlabel("Cluster")
    axes[0].legend(title="")

    # Mois
    sns.heatmap(
        comp_month,
        annot=True,
        fmt=".0f",
        cmap="YlOrRd",
        ax=axes[1]
    )

    axes[1].set_title("Répartition mensuelle par cluster (%)")
    axes[1].set_xlabel("Mois")
    axes[1].set_ylabel("Cluster")

    fig.suptitle(titre, fontsize=14)

    plt.tight_layout()
    plt.show()

    return comp_weekend, comp_month
plot_cluster_calendar_composition(results_amplet, "Composition temporelle des clusters(amplet)")
plot_cluster_calendar_composition(results_forme, "Composition temporelle des clusters(forme)")


In [ ]:
def plot_cluster_timeline(results, k_final, titre):
    results_sorted = results.sort_values("date").reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(16, 3))
    scatter = ax.scatter(
        results_sorted["date"], [1] * len(results_sorted),
        c=results_sorted["cluster"], cmap="tab10", s=15
    )
    ax.set_yticks([])
    ax.set_title(titre)
    plt.colorbar(scatter, ax=ax, label="Cluster", ticks=range(k_final))
    plt.tight_layout()
    plt.show()

plot_cluster_timeline(results_forme, k_forme, "Assignation de cluster au fil de l'année (forme)")
plot_cluster_timeline(results_amplet, k_amplet, "Assignation de cluster au fil de l'année (amplitude)")

## PCA sur les profils 

In [ ]:


def compute_pca(X, n_components=10, titre1=""):
    """
    Applique une PCA et affiche la variance expliquée.
    X : (ndarray) Matrice des données.
    n_components : (int) Nombre de composantes principales.
    X_pca : (ndarray)Données projetées dans l'espace PCA.
    pca : (PCA)Objet PCA entraîné.
    var_ratio : (ndarray) Variance expliquée par chaque composante.
    var_cum : (ndarray)Variance expliquée cumulée.
    n_components_90 : (int)Nombre de composantes nécessaires pour expliquer 90 % de la variance.
    """

    pca = PCA(n_components=n_components, random_state=42)
    X_pca = pca.fit_transform(X)

    var_ratio = pca.explained_variance_ratio_
    var_cum = np.cumsum(var_ratio)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(range(1, len(var_ratio) + 1), var_ratio)
    axes[0].set_xlabel("Composante")
    axes[0].set_ylabel("Variance expliquée")
    axes[0].set_title(f"Variance expliquée par composante {titre1}")

    axes[1].plot(range(1, len(var_cum) + 1), var_cum, marker="o")
    axes[1].axhline(0.90, color="red", linestyle="--", label="90 %")
    axes[1].set_xlabel("Nombre de composantes")
    axes[1].set_ylabel("Variance cumulée")
    axes[1].legend()
    axes[1].set_title(f"Variance cumulée {titre1}")

    plt.tight_layout()

  
    plt.show()

    n_components_90 = np.argmax(var_cum >= 0.90) + 1

    print(f"Composantes nécessaires pour 90 % de variance {titre1}: {n_components_90}")

    return X_pca, pca, var_ratio, var_cum, n_components_90

X_pca_forme, pca_forme, var_ratio_forme, var_cum_forme, n_components_90_forme=compute_pca(X, n_components=10, titre1="(Forme)")
Y_pca_amplet, pca_amplet, var_ratio_amplet, var_cum_amplet, n_components_90_amplet=compute_pca(Y, n_components=10, titre1="(Amplet)")

In [ ]:
def plot_pca_clusters(X_pca, cluster_labels, var_ratio, title="Projection PCA des clusters"):
    """Affiche la projection 2D de la PCA colorée par les clusters."""
    fig, ax = plt.subplots(figsize=(8, 7))

    scatter = ax.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=cluster_labels,
        cmap="tab10",
        s=8,
        alpha=0.6
    )

    ax.set_xlabel(f"PC1 ({var_ratio[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({var_ratio[1]*100:.1f}%)")
    ax.set_title(title)

    plt.colorbar(scatter, ax=ax, label="Cluster")
    plt.tight_layout()
    plt.show()
    return 0

plot_pca_clusters(X_pca_forme, cluster_labels_forme, var_ratio_forme, title="Projection PCA des clusters (forme)")
plot_pca_clusters(Y_pca_amplet, cluster_labels_amplet, var_ratio_amplet, title="Projection PCA des clusters (amplet)")

## Bâtiment de comparaison : 10672-0

`10672-0` est le bâtiment de référence utilisé dans `timeseries_clustering_multi.ipynb` (sélectionné automatiquement parmi les 100 bâtiments tout-électriques). On reproduit ici exactement le même pipeline mono-bâtiment que pour `347201-0` ci-dessus, pour pouvoir comparer les deux côte à côte.

In [ ]:
BLDG_2_PATH = DATA_RAW / "timeseries_100_all_electric" / "10672-0.parquet"

df_2 = pd.read_parquet(BLDG_2_PATH)

ts_2 = df_2[COL].values
dates_2 = pd.to_datetime(df_2["timestamp"]) if "timestamp" in df_2.columns else pd.to_datetime(df_2.index)

n_days_2 = len(ts_2) // steps_per_day
ts_2 = ts_2[: n_days_2 * steps_per_day]
dates_2 = dates_2[: n_days_2 * steps_per_day]

days_2 = ts_2.reshape(n_days_2, steps_per_day)
day_dates_2 = (
    pd.Series(dates_2)
    .groupby(pd.Series(np.arange(len(dates_2))) // steps_per_day)
    .first()
    .dt.normalize()
    .values
)

print("Nombre de jours :", n_days_2)
print("Shape matrice jours :", days_2.shape)
print("Première date :", day_dates_2[0])

In [ ]:
# Supprimer les jours avec NaN
mask_valid_2 = ~np.isnan(days_2).any(axis=1)
print(f"Jours invalides supprimés : {(~mask_valid_2).sum()} / {len(days_2)}")

days_clean_2 = days_2[mask_valid_2]
day_dates_clean_2 = day_dates_2[mask_valid_2]

Même logique de normalisation qu'expliqué plus haut (*Option A — StandardScaler global* pour l'amplitude, *Option B — z-score intra-jour* pour la forme), appliquée ici au second bâtiment de comparaison (10672-0).

In [ ]:
# Normalisation : forme (z-score intra-jour) et amplitude (StandardScaler global)
scaler_global_2 = StandardScaler()
days_scaled_global_2 = scaler_global_2.fit_transform(days_clean_2)

day_mean_2 = days_clean_2.mean(axis=1, keepdims=True)
day_std_2  = days_clean_2.std(axis=1, keepdims=True)
day_std_2[day_std_2 == 0] = 1e-8

days_scaled_shape_2 = (days_clean_2 - day_mean_2) / day_std_2

X2 = days_scaled_shape_2
Y2 = days_scaled_global_2

In [ ]:
k_forme_2 = plot_cluster_profiles(
    X2, "Méthode du coude (inertie) - 10672-0 (forme)", "Silhouette score - 10672-0 (forme)"
)
k_amplet_2 = plot_cluster_profiles(
    Y2, "Méthode du coude (inertie) - 10672-0 (amplitude)", "Silhouette score - 10672-0 (amplitude)"
)

In [ ]:
kmeans_forme_2, cluster_labels_forme_2, results_forme_2 = cluster_days(k_forme_2, X_input=X2, dates_input=day_dates_clean_2)
kmeans_amplet_2, cluster_labels_amplet_2, results_amplet_2 = cluster_days(k_amplet_2, X_input=Y2, dates_input=day_dates_clean_2)
print(results_forme_2)
print(results_amplet_2)

In [ ]:
plot_interactive_cluster_profiles(
    cluster_labels=cluster_labels_amplet_2,
    k_final=k_amplet_2,
    titre="10672-0 — Profil journalier par rapport à la quantité d'énergie consommée",
    days_input=days_clean_2,
)
plot_interactive_cluster_profiles(
    cluster_labels=cluster_labels_forme_2,
    k_final=k_forme_2,
    titre="10672-0 — Profil journalier par rapport à la forme d'énergie consommée",
    days_input=days_clean_2,
)

In [ ]:
enduse_days_2 = extract_enduse_days(df_2, mask_valid_2, n_days_2)
print("Postes de consommation détectés (10672-0) :", list(enduse_days_2.keys()))

plot_cluster_profiles_by_enduse(
    enduse_days_2, cluster_labels_forme_2, k_forme_2,
    "10672-0 — Décomposition par poste — profils de forme"
)
plot_cluster_profiles_by_enduse(
    enduse_days_2, cluster_labels_amplet_2, k_amplet_2,
    "10672-0 — Décomposition par poste — profils d'amplitude"
)

In [ ]:
plot_cluster_calendar_composition(results_amplet_2, "Composition temporelle des clusters (amplet) — 10672-0")
plot_cluster_calendar_composition(results_forme_2, "Composition temporelle des clusters (forme) — 10672-0")

In [ ]:
plot_cluster_timeline(results_forme_2, k_forme_2, "Assignation de cluster au fil de l'année (forme) — 10672-0")
plot_cluster_timeline(results_amplet_2, k_amplet_2, "Assignation de cluster au fil de l'année (amplitude) — 10672-0")

In [ ]:
X_pca_forme_2, pca_forme_2, var_ratio_forme_2, var_cum_forme_2, n_components_90_forme_2 = compute_pca(
    X2, n_components=10, titre1="(Forme, 10672-0)"
)
Y_pca_amplet_2, pca_amplet_2, var_ratio_amplet_2, var_cum_amplet_2, n_components_90_amplet_2 = compute_pca(
    Y2, n_components=10, titre1="(Amplet, 10672-0)"
)

plot_pca_clusters(X_pca_forme_2, cluster_labels_forme_2, var_ratio_forme_2, title="Projection PCA des clusters (forme) — 10672-0")
plot_pca_clusters(Y_pca_amplet_2, cluster_labels_amplet_2, var_ratio_amplet_2, title="Projection PCA des clusters (amplet) — 10672-0")

In [ ]:
def build_day_features(
    parquet_path,
    col=COL,
    weather_cols=[
        "out.outdoor_air_drybulb_temp..c",
        "out.outdoor_air_relative_humidity..percentage",
        "out.outdoor_air_wetbulb_temp..c",
        "out.outdoor_humidity_ratio..kgwater_per_kgdryair",
    ],
):

    d = pd.read_parquet(parquet_path)
    ts = d[col].values
    n_days_ = len(ts) // steps_per_day
    ts = ts[: n_days_ * steps_per_day]

    dates_ = pd.to_datetime(d["timestamp"]) if "timestamp" in d.columns else pd.to_datetime(d.index)
    dates_ = dates_[: n_days_ * steps_per_day]

    days_ = ts.reshape(n_days_, steps_per_day)
    dates_daily = (
        pd.Series(dates_)
        .groupby(pd.Series(np.arange(len(dates_))) // steps_per_day)
        .first()
        .dt.normalize()
        .values
    )

    mask_valid_ = ~np.isnan(days_).any(axis=1)
    days_ = days_[mask_valid_]
    dates_daily = dates_daily[mask_valid_]

    # -------- Forme --------
    mean_ = days_.mean(axis=1, keepdims=True)
    std_ = days_.std(axis=1, keepdims=True)
    std_[std_ == 0] = 1e-8
    shape_ = (days_ - mean_) / std_

    # -------- Amplitude --------
    amplitude_ = days_.sum(axis=1)

    # -------- Métadonnées --------
    meta = {
        "bldg_id": Path(parquet_path).stem,
        "date": dates_daily,
        "amplitude": amplitude_,
    }

    # -------- Variables météo --------
    for wcol in weather_cols:

        if wcol in d.columns:
            weather = (
                d[wcol]
                .values[: n_days_ * steps_per_day]
                .reshape(n_days_, steps_per_day)[mask_valid_]
            )

            prefix = wcol.replace("out.", "").replace("..", "_").replace(".", "_")

            meta[f"{prefix}_mean"] = weather.mean(axis=1)
            meta[f"{prefix}_min"] = weather.min(axis=1)
            meta[f"{prefix}_max"] = weather.max(axis=1)

        else:
            prefix = wcol.replace("out.", "").replace("..", "_").replace(".", "_")

            meta[f"{prefix}_mean"] = np.nan
            meta[f"{prefix}_min"] = np.nan
            meta[f"{prefix}_max"] = np.nan

    return pd.DataFrame(meta), shape_



In [ ]:
# ============================
# 1) Extraction de tous les bâtiments
# ============================

parquet_files = sorted(glob.glob(str(DATA_RAW / "*.parquet")))

all_meta = []
all_shapes = []

for f in parquet_files:
    meta_f, shape_f = build_day_features(f)
    all_meta.append(meta_f)
    all_shapes.append(shape_f)

meta_all = pd.concat(all_meta, ignore_index=True)
shape_all = np.vstack(all_shapes)

print("Tous les bâtiments")
print("meta :", meta_all.shape)
print("shape:", shape_all.shape)
print("Nombre bâtiments :", meta_all["bldg_id"].nunique())


# ============================
# 2) Sélection d'un bâtiment
# ============================

bldg_selected = "347201-0"

mask_bldg = meta_all["bldg_id"].values == bldg_selected

meta_bldg = meta_all[mask_bldg].reset_index(drop=True)
shape_bldg = shape_all[mask_bldg]


print("\nBâtiment sélectionné :", bldg_selected)
print("meta :", meta_bldg.shape)
print("shape:", shape_bldg.shape)



# ============================
# 3) Sélection des variables météo
#    (SUR CE bâtiment uniquement)
# ============================

weather_feature_cols = [
    c for c in meta_bldg.columns
    if c.endswith(("_mean", "_min", "_max"))
]

print("\nColonnes météo utilisées :")
print(weather_feature_cols)



# ============================
# 4) Suppression des jours incomplets
# ============================

mask_ok = (
    meta_bldg[weather_feature_cols]
    .notna()
    .all(axis=1)
    .values
)

print(
    f"\nJours conservés météo complète : "
    f"{mask_ok.sum()} / {len(mask_ok)}"
)


shape_ok = shape_bldg[mask_ok]

meta_ok = (
    meta_bldg[mask_ok]
    .reset_index(drop=True)
)



# ============================
# 5) Construction des features scalaires (amplitude + météo, PAS de PCA dessus)
# ============================

# amplitude énergie
amp_log = np.log1p(
    meta_ok["amplitude"].values
)


scalar_features = np.column_stack(
    [
        amp_log,
        *[
            meta_ok[c].values
            for c in weather_feature_cols
        ]
    ]
)


# Amplitude + météo : standardisées globalement, gardées TELLES QUELLES (pas de PCA),
# pour rester interprétables dans les analyses de corrélation qui suivent.
scalar_scaled = StandardScaler().fit_transform(
    scalar_features
)

print("\nFeatures scalaires (amplitude + météo, sans PCA) :", scalar_scaled.shape)

In [ ]:
# PCA appliquée UNIQUEMENT sur la forme (shape_ok) : la météo et l'amplitude
# restent brutes (standardisées) et ne passent jamais par la PCA.
shape_pca_multi, pca_shape_multi, var_ratio_shape_multi, var_cum_shape_multi, _ = compute_pca(
    shape_ok, n_components=10, titre1="(Forme, multivarié)"
)

# Concaténation : forme réduite par PCA + amplitude/météo brutes standardisées (PAS de PCA sur la météo)
X_multi = np.hstack([shape_pca_multi, scalar_scaled]).astype(np.float64)
print("X_multi :", X_multi.shape)

k_multi_opt = plot_cluster_profiles(
    X_multi,
    "Méthode du coude - Clustering multivarié (forme+amplitude+météo)",
    "Silhouette - Clustering multivarié (forme+amplitude+météo)"
)

In [ ]:
kmeans_multi = KMeans(n_clusters=k_multi_opt, random_state=42, n_init=20)
cluster_labels_multi = kmeans_multi.fit_predict(X_multi)

meta_ok["cluster_multi"] = cluster_labels_multi
meta_ok["weekday"] = pd.to_datetime(meta_ok["date"]).dt.day_name()
meta_ok["is_weekend"] = pd.to_datetime(meta_ok["date"]).dt.dayofweek >= 5
meta_ok["month"] = pd.to_datetime(meta_ok["date"]).dt.month

print(meta_ok["cluster_multi"].value_counts().sort_index())

# Les axes affichés sont PC1/PC2 de la FORME uniquement (pas de PCA sur la météo/amplitude)
plot_pca_clusters(
    X_multi, cluster_labels_multi, var_ratio_shape_multi,
    title=f"Clustering multivarié (forme+amplitude+météo), k={k_multi_opt}"
)

In [ ]:
# On prend la première colonne météo "_mean" comme référence principale pour les plots (généralement la température)
main_weather_col = next((c for c in weather_feature_cols if c.endswith("_mean")), weather_feature_cols[0])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=meta_ok, x="cluster_multi", y=main_weather_col, ax=axes[0])
axes[0].set_title(f"{main_weather_col} par cluster")

sns.boxplot(data=meta_ok, x="cluster_multi", y="amplitude", ax=axes[1])
axes[1].set_title("Amplitude (kWh/jour) par cluster")

comp_weekend_multi = pd.crosstab(meta_ok["cluster_multi"], meta_ok["is_weekend"], normalize="index") * 100
comp_weekend_multi.columns = ["Semaine (%)", "Weekend (%)"]
comp_weekend_multi.plot(kind="bar", stacked=True, ax=axes[2], colormap="Set2")
axes[2].set_title("Semaine / weekend par cluster")

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap complète : toutes les variables météo moyennes par cluster (vue d'ensemble)
mean_cols = [c for c in weather_feature_cols if c.endswith("_mean")] + ["amplitude"]
cluster_summary = meta_ok.groupby("cluster_multi")[mean_cols].mean()

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(cluster_summary.T, annot=True, fmt=".1f", cmap="coolwarm", ax=ax)
ax.set_title("Moyenne des variables par cluster (résumé)")
plt.tight_layout()
plt.show()

In [ ]:
# Profils de forme moyens par cluster
fig, ax = plt.subplots(figsize=(12, 7))
hours = np.linspace(0, 24, steps_per_day)
colors = plt.cm.tab10(np.linspace(0, 1, k_multi_opt))

for c in range(k_multi_opt):
    mask = cluster_labels_multi == c
    mean_shape = shape_ok[mask].mean(axis=0)
    ax.plot(hours, mean_shape, label=f"Cluster {c} (n={mask.sum()})", color=colors[c], linewidth=2)

ax.set_xlabel("Heure")
ax.set_ylabel("Consommation normalisée (forme)")
ax.set_title("Profils de forme moyens par cluster multivarié")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
'''
meta_ok.to_parquet(DATA_PROCESSED / "clustering_multivarie_jours_types.parquet", index=False)

import joblib
joblib.dump(kmeans_multi, DATA_PROCESSED / "kmeans_multivarie.joblib")
joblib.dump(pca_shape_multi, DATA_PROCESSED / "pca_shape_multivarie.joblib")

meta_ok.head()

'''